# Beyond DAGs: the three failures, and what answers each

A plain DAG assumes four things at once: no latent confounding, no cycles, no
potential outcomes on the picture, and one population with complete data. Each
assumption has a graphical framework that repairs it. Three of them live here.

| the DAG assumes | it fails when | axiom answers with |
|---|---|---|
| causal sufficiency | something you did not measure drives two variables | **ADMGs + the ID algorithm** — `identify_effect` |
| acyclicity | quantities are determined together, or feed back | **sigma-separation** — `MixedGraph`, `acyclify` |
| the estimand is obvious | you owe a stakeholder an ATE, not a path claim | **SWIGs** — `swig`, `single_world_ignorability` |
| one population, complete data | you generalize, or data is missing | selection diagrams (`transport_verdict`); m-graphs are not built |

The fourth row is honest about a gap: transport is here, missing-data graphs
are not. See `docs/notes/0012` for what else was deliberately left out.

In [ ]:
from axiom.core import D, Param, dimensionless
from axiom.dynamics import Variable, parse_system
from axiom.identify import (
    CausalGraph, Density, Formula, Hedge, IdentifiedEffect, JointTable, Marginal, MixedGraph,
    Product, Ratio, acyclify, admissible_set_exists, districts, identify,
    identify_conditional_effect, identify_effect, latent_projection, sequential_ignorability,
    sequential_plan, sigma_separated, single_world_ignorability, swig, unrolled_graph,
    unrolled_mixed_graph,
)

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

NONE = dimensionless()

## 1. Latent confounding: a search versus a proof

`identify` searches a menu — back-door, front-door, instrument — and reports
the first route that works. When it finds nothing it says so honestly, and
that report is **not a proof**: an effect can be identifiable with no named
route applying.

Here is a graph where exactly that happens.

In [ ]:
graph = CausalGraph.from_edges("M -> Y, W -> M, W -> X, W -> Y, X -> M, M <-> W, X <-> Y")
menu = identify(graph, "X", "Y")
print("route menu :", menu.verdict.status, "|", menu.verdict.reason)

result: IdentifiedEffect = identify_effect(graph, "X", "Y")
print("ID         :", result.verdict.status, "|", result.to_text())

The ID algorithm (Tian & Pearl 2002; Shpitser & Pearl 2006) is sound **and
complete**: it returns an estimand when one exists and a *hedge* when none
does. A hedge is a proof of non-identifiability, not a failed search — which
matters, because the two have opposite consequences. One says look harder; the
other says go and run an experiment.

In [ ]:
bow = CausalGraph.from_edges("X -> Y, U -> X, U -> Y", unmeasured=["U"])
blocked = identify_effect(bow, "X", "Y")
print("identified:", blocked.identified)
hedge: Hedge = blocked.hedge
print("hedge     :", hedge.root, hedge.subset)
print(blocked.verdict.reason)

### The estimand is an object, not a sentence

What comes back is a `Formula` over the observational distribution: `Density`,
`Product`, `Marginal` and `Ratio`. It renders, and — the part that makes the
claim checkable — it *evaluates* against a discrete joint.

In [ ]:
back_door = identify_effect(CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y"), "X", "Y")
front_door = identify_effect(
    CausalGraph.from_edges("X -> M, M -> Y, U -> X, U -> Y", unmeasured=["U"]), "X", "Y"
)
print("back door :", back_door.to_text())
print("front door:", front_door.to_text())

formula: Formula = front_door.formula
print()
print("node types:", sorted({type(n).__name__ for n in [formula]}))
print("is it a Marginal of a Product of Density terms?", isinstance(formula, Marginal))

In [ ]:
import numpy as np

# A world with a known truth: Z -> X -> Y and Z -> Y, all binary.
rng = np.random.default_rng(0)
levels = 2
p_z = rng.dirichlet([1, 1])
p_x = rng.dirichlet([1, 1], size=levels)          # X | Z
p_y = rng.dirichlet([1, 1], size=(levels, levels))  # Y | X, Z

observational = np.zeros((levels, levels, levels))  # (X, Y, Z)
for z in range(levels):
    for x in range(levels):
        for y in range(levels):
            observational[x, y, z] = p_z[z] * p_x[z, x] * p_y[x, z, y]
joint = JointTable(names=("X", "Y", "Z"), table=observational)

from axiom.identify.formula import evaluate

got = np.asarray(evaluate(back_door.formula, joint))
truth = np.array([[sum(p_z[z] * p_y[x, z, y] for z in range(levels)) for y in range(levels)]
                  for x in range(levels)])
print("estimand :", np.round(got.reshape(levels, levels), 6))
print("truth    :", np.round(truth, 6))
print("agree    :", np.allclose(got.reshape(levels, levels), truth))

### The machinery underneath

`latent_projection` turns explicit latents into bidirected edges, so a graph
drawn either way gives the same answer. `districts` are the C-components — the
sets a latent cause reaches together, which is what the algorithm recurses on.

In [ ]:
explicit = CausalGraph.from_edges("X -> M, M -> Y, U -> X, U -> Y", unmeasured=["U"])
admg = latent_projection(explicit)
print("projected:", admg.to_text())
print("districts:", districts(admg))
print("same estimand as the bidirected spelling:",
      identify_effect(explicit, "X", "Y").formula
      == identify_effect(CausalGraph.from_edges("X -> M, M -> Y, X <-> Y"), "X", "Y").formula)

conditional = identify_conditional_effect(
    CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y, X -> W, W -> Y"), "X", "Y", ["W"]
)
print()
print("conditional effect:", conditional.to_text()[:80])
print("IDC moved into the do:", conditional.detail["moved_into_do"])

## 2. Cycles: d-separation is unsound, so use the criterion that is not

Conditioning on a variable inside a feedback loop does **not** block the flow
of information around the loop — the loop's members are functionally
intertwined. So a d-separation claim read off a cyclic diagram can be false.

Sigma-separation (Forré & Mooij 2017) differs from d-separation in one place:
*a non-collider blocks only if it points to a node in a different strongly
connected component.*

In [ ]:
loop = MixedGraph.from_edges("x -> a, a -> b, b -> a, b -> y")
print("loops:", loop.cyclic_components)
print("x and y separated given a?  sigma:", sigma_separated(loop, "x", "y", ["a"]))
print("                       (d, wrongly):",
      CausalGraph.from_edges("x -> a, a -> b, b -> y").d_separated("x", "y", ["a"]))
print("x and y separated given b?  sigma:", sigma_separated(loop, "x", "y", ["b"]))

`a` points at `b`, inside its own loop, so conditioning on it cannot block.
`b` points at `y`, outside the loop, so conditioning on it does.

The **acyclification** (Mooij & Claassen 2020) turns the cyclic graph into an
acyclic one whose d-separation *is* the original's sigma-separation: every
loop becomes a bidirected clique and every parent of a loop points at all of
it. Read causally, that is the reduced form.

In [ ]:
print("acyclified:", acyclify(loop).to_text())

In [ ]:
market = parse_system(
    """
    quantity = a - b * price + c * income
    price    = d + e * quantity + cost
    """,
    variables=(
        Variable(name="quantity", dimension=D.outcome),
        Variable(name="price", dimension=D.currency),
        Variable(name="income", dimension=D.currency, role="exogenous"),
        Variable(name="cost", dimension=D.currency, role="exogenous"),
    ),
    parameters=(
        Param(name="a", dimension=D.outcome),
        Param(name="b", dimension=D.outcome / D.currency),
        Param(name="c", dimension=D.outcome / D.currency),
        Param(name="d", dimension=D.currency),
        Param(name="e", dimension=D.currency / D.outcome),
    ),
    name="market",
)
structural = unrolled_mixed_graph(market, periods=1)
reduced = unrolled_graph(market, periods=1)
print("as written  :", structural.to_text(), "| cyclic:", not structural.is_acyclic)
print("reduced form:", reduced.to_text())
print()
print("are quantity and price independent given everything feeding the block?")
print("  ", reduced.d_separated("quantity.t0", "price.t0", ["cost.t0", "income.t0"]))

**No** — and the bidirected edge is what says so. Two quantities determined
together share whatever disturbs the block; a reduced form with shared parents
but no bidirected edge would wrongly report them independent. This is the
acyclification, so it is not a convention chosen here — it is the object whose
d-separation answers sigma-separation on the graph as written.

## 3. SWIGs: put the potential outcome on the graph

A DAG does not display potential outcomes, and the usual way of reading one
quietly asserts *cross-world* independences that no experiment can check. A
SWIG (Richardson & Robins 2013) splits each intervened node into a random half
that keeps its parents and a fixed half that emits its children; everything
downstream becomes ``Y(x)``.

In [ ]:
g = CausalGraph.from_edges("Z -> X, Z -> Y, X -> M, M -> Y")
print("original:", g.to_text())
print("swig    :", swig(g, ["X"]).to_text())

In [ ]:
# Ignorability, stated about the potential outcome, agrees with the back-door criterion.
table(
    [
        [str(adjustment or "{}"), str(single_world_ignorability(g, "X", "Y", adjustment))]
        for adjustment in ([], ["Z"], ["M"])
    ],
    headers=("conditioning on", "Y(x) independent of X"),
)

Adjusting for `M` is refused, and for the right reason: in the SWIG `M` became
`M(x)`, a variable in the hypothetical world. Conditioning on it is not
something data can do.

For a **sequential** regime every stage splits at once, and the independences
the g-formula needs are read off one picture. That is the same claim
`sequential_plan` checks with the sequential back-door criterion — two
derivations, held against each other.

In [ ]:
system = parse_system(
    """
    outcome = beta * dose + rho * outcome[t-1] + kappa * frailty
    dose    = phi * outcome[t-1] + protocol
    """,
    variables=(
        Variable(name="outcome", dimension=D.outcome),
        Variable(name="dose", dimension=D.currency),
        Variable(name="protocol", dimension=D.currency, role="exogenous"),
        Variable(name="frailty", dimension=NONE, role="exogenous", observed=False),
    ),
    parameters=(
        Param(name="beta", dimension=D.outcome / D.currency),
        Param(name="rho", dimension=NONE),
        Param(name="kappa", dimension=D.outcome),
        Param(name="phi", dimension=D.currency / D.outcome),
    ),
    name="feedback",
)
graph = unrolled_graph(system, periods=3)
stages = ["dose.t0", "dose.t1", "dose.t2"]
plan = sequential_plan(graph, stages, "outcome.t2")
print("g-formula plan:", plan.adjustments)
print("SWIG agrees   :", sequential_ignorability(graph, stages, "outcome.t2", plan.adjustments))
print("on a wrong plan:", sequential_ignorability(graph, stages, "outcome.t2",
                                                  [("outcome.t1",), (), ()])[1][:88])

## What is still a DAG's job

None of this replaces the DAG; it extends what can be drawn and still answered.
`admissible_set_exists` and the back-door search remain the fastest answer when
they apply, and the ID algorithm returns exactly the adjustment formula in that
case — as the first example showed. What changed is that "no route found" is no
longer the end of the conversation.

In [ ]:
simple = CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y")
print("back-door set exists:", admissible_set_exists(simple, "X", "Y"))
print("and ID returns it   :", identify_effect(simple, "X", "Y").to_text())
ratio_example: Ratio = Ratio(
    numerator=Density(outcomes=("X", "Y")), denominator=Density(outcomes=("X",))
)
product_example: Product = Product(
    factors=(Density(outcomes=("Y",), given=("X",)), Density(outcomes=("X",)))
)
print("the algebra:", ratio_example, "|", type(product_example).__name__)

## 4. Granularity: cluster-DAGs

The fourth DAG assumption is that you know the graph at the granularity you
are asking about. Usually you do not — but you often know how *blocks* of
variables relate. A cluster-DAG (Anand, Ribeiro, Tian & Bareinboim 2023) writes
down exactly that, and its semantics is about **absence**: a missing edge
between two clusters says no variable in one is adjacent to any variable in the
other. Within a cluster, anything goes.

Both d-separation and identification transfer: an answer read off the cluster
graph holds in *every* compatible micro-level DAG.

In [ ]:
from axiom.identify import ClusterDAG, compatible, identify_cluster_effect

market = ClusterDAG(
    clusters={"cost": ("wage", "fuel"), "price": ("price",), "demand": ("units",)},
    edges=(("cost", "price"), ("price", "demand")),
    name="a two-block market",
)
print("clusters   :", {c: market.members(c) for c in sorted(market.clusters)})
print("variables  :", market.variables)
print("singletons :", market.singletons)
print("cost indep demand given price?", market.d_separated("cost", "demand", ["price"]))
print()
print("effect of the cost block on demand:", identify_cluster_effect(market, "cost", "demand").to_text())

In [ ]:
# Compatibility is what the cluster claim rules out.
micro = CausalGraph.from_edges("wage -> fuel, fuel -> price, price -> units")
print("compatible:", compatible(market, micro))
wrong_way = CausalGraph.from_edges("wage -> fuel, price -> fuel, price -> units")
print("an edge running the wrong way across clusters:", compatible(market, wrong_way))

# And coarsening a graph you *do* know gives the C-DAG a reader would be told.
known = CausalGraph.from_edges("wage -> price, fuel -> price, price -> units")
print("coarsened:", ClusterDAG.from_dag(known, {c: market.members(c) for c in market.clusters}).edges)

In [ ]:
# The refusal that keeps a cluster answer honest: a variable-level question
# is not one a cluster-level graph contains the structure to answer.
try:
    identify_cluster_effect(market, "wage", "demand")
except Exception as e:
    print(type(e).__name__, "->", e)

That refusal is the framework earning its keep. An aggregate that behaves like
a variable is a *claim*; a cluster-DAG is where the claim gets stated instead
of assumed, and where asking past it gets refused rather than silently
answered at the wrong granularity.

## Seeing it

`enable()` at the top of this notebook already made a bare result on the last
line of a cell render itself — a card drawn by `rich`, or the same content as
aligned plain text where `rich` is not installed. `show` does it on demand, for
a result that is not the last thing in its cell.

`axiom.viz` draws the figure this subpackage's results are actually about.

In [ ]:
from axiom.display import show
from axiom.identify import CausalGraph, identify
from axiom.viz import causal_graph

seen = CausalGraph.from_edges(
    "age -> dose, age -> pressure, dose -> adherence, adherence -> pressure, "
    "u -> dose, u -> pressure",
    unmeasured=["u"],
    name="a trial with one thing unmeasured",
)
show(identify(seen, "dose", "pressure"))
causal_graph(seen)

The confounder `u` is unmeasured and there is no adjustment set that blocks `dose ← u → pressure`, so the back door is shut. The effect is identified anyway — by the front door, through `adherence`, because nothing points into the mediator from outside. The card names the route; the figure is how you check that the mediator really is clean.